# Pelatihan dan evaluasi IndoBERT ABSA SATUSEHAT

1. Unggah notebook ini melalui **File → Upload notebook** di Google Colab.
2. Pilih **Runtime → Change runtime type → T4 GPU**, jika tersedia.
3. Jalankan semua sel secara berurutan. Notebook akan berhenti jika GPU tidak tersedia.
4. Unduh ZIP hasil, lalu berikan ZIP tersebut untuk memperbarui laporan.

Default: IndoBERT **base**, seed **42, 43, 44**, 400 ulasan latih,
100 validasi, dan evaluasi akhir pada 150 ulasan berlabel manual yang sama.
Parameter dipilih dengan validasi; label uji hanya digunakan saat evaluasi akhir.
Untuk IndoBERT large, aktifkan `RUN_LARGE` sebelum mulai; hasilnya dipisahkan.

Notebook ini belum dieksekusi. Tidak ada skor IndoBERT yang diklaim tersedia.
Kode dan data diambil dari commit proyek yang dikunci agar dapat ditelusuri.
Hasil disimpan setelah setiap seed selesai; jika runtime terputus di tengah
epoch, seed yang belum selesai perlu diulang. Berkas runtime Colab bersifat sementara.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import torch

assert torch.cuda.is_available(), "GPU belum aktif. Pilih Runtime > Change runtime type > T4 GPU, lalu jalankan ulang."
print("GPU:", torch.cuda.get_device_name(0))
SOURCE_COMMIT = "f8f8f2b7dbd986788c68acf2a9b0aa246b94c15c"
REPO_URL = "https://github.com/bambangh31/web-mining-absa-kelompok-2.git"
REPO = Path("/content/web-mining-absa")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if actual != SOURCE_COMMIT:
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_COMMIT], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-model.txt"], check=True)
sys.path.insert(0, str(REPO))
print("Sumber:", SOURCE_COMMIT)


In [ ]:
import csv
import hashlib
import json
from datetime import datetime, timezone
import importlib.metadata

RUN_LARGE = False
SEEDS = [42, 43, 44]
MAX_EPOCHS = 10
OUT = Path("/content/hasil_indobert")
OUT.mkdir(parents=True, exist_ok=True)
from src.model_data import Dataset, prediction_ids

dataset = Dataset("data/processed/reviews_clean.csv", "data/annotations/split.csv", "data/annotations/verified.csv")
test_ids = prediction_ids(dataset, predict_from="data/annotations/final_v1/gold.csv")
assert len(dataset.train_ids) == 400 and len(dataset.val_ids) == 100 and len(test_ids) == 150
assert not set(test_ids) & (set(dataset.train_ids) | set(dataset.val_ids))
assert all(dataset.split[rid] == "gold_test" for rid in test_ids)
sources = ["data/processed/reviews_clean.csv", "data/annotations/split.csv",
           "data/annotations/verified.csv", "data/annotations/final_v1/gold.csv"]
manifest = {
    "source_commit": SOURCE_COMMIT, "seeds": SEEDS, "max_epochs": MAX_EPOCHS,
    "variants": ["base", "large"] if RUN_LARGE else ["base"],
    "n_train": 400, "n_validation": 100, "n_test": 150,
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "gpu": torch.cuda.get_device_name(0), "python": sys.version,
    "packages": {n: importlib.metadata.version(n) for n in ["torch", "transformers", "numpy"]},
    "sha256_sources": {p: hashlib.sha256(Path(p).read_bytes()).hexdigest() for p in sources},
}
manifest_path = OUT / "manifest.json"
if manifest_path.exists():
    old = json.loads(manifest_path.read_text())
    for key in ["source_commit", "seeds", "max_epochs", "variants", "sha256_sources"]:
        assert old[key] == manifest[key], "Konfigurasi berubah: gunakan folder OUT baru."
else:
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Latih 400, validasi 100, uji manual 150. Sumber data telah dicatat.")


In [ ]:
import shutil
from google.colab import files
from src.absa_schema import ASPECTS

for variant in manifest["variants"]:
    model_code = "B2" if variant == "base" else "B2-L"
    for seed in SEEDS:
        pred_path = OUT / "predictions" / f"{model_code}_{seed}.csv"
        metrics_path = OUT / "metrics" / f"{model_code}_{seed}.json"
        complete = False
        if pred_path.exists() and metrics_path.exists():
            saved = json.loads(metrics_path.read_text())
            with pred_path.open(encoding="utf-8-sig", newline="") as stream:
                rows = list(csv.DictReader(stream))
            complete = (saved["seed"] == seed and saved["model_code"] == model_code
                        and saved["config"]["max_epochs"] == MAX_EPOCHS
                        and saved["config"]["n_latih"] == 400
                        and len(rows) == 150 and {r["reviewId"] for r in rows} == set(test_ids)
                        and all(a in r and r[a] for r in rows for a in ASPECTS))
        if complete:
            print(f"Lewati {model_code} seed {seed}: hasil lengkap sudah tersedia.")
            continue
        subprocess.run([
            sys.executable, "-u", "src/model_indobert.py", "--variant", variant,
            "--seeds", str(seed), "--labels", "data/annotations/verified.csv",
            "--predict-from", "data/annotations/final_v1/gold.csv",
            "--max-epochs", str(MAX_EPOCHS), "--out", str(OUT),
        ], check=True)
        print(f"Selesai: {model_code}, seed {seed}")
        archive = shutil.make_archive(f"/content/hasil_indobert_{model_code}_{seed}", "zip", OUT)
        files.download(archive)


In [ ]:
from src.evaluate import evaluate, load_labels, average_runs

pred_args = [
    "B1-LR=results/predictions/B1-LR_42.csv",
    "B1-SVM=results/predictions/B1-SVM_42.csv",
    "B1-NB=results/predictions/B1-NB_42.csv",
    "B3-Gemini=data/annotations/llm_gemini-3.5-flash-lite.csv",
]
gold = load_labels("data/annotations/final_v1/gold.csv")
summary = {}
for variant in manifest["variants"]:
    model_code = "B2" if variant == "base" else "B2-L"
    scores = []
    for seed in SEEDS:
        path = OUT / "predictions" / f"{model_code}_{seed}.csv"
        pred = load_labels(path)
        assert set(pred) == set(test_ids), f"ID prediksi tidak sesuai: {path}"
        scores.append(evaluate(gold, pred, test_ids))
        pred_args.append(f"{model_code}_{seed}={path}")
    summary[model_code] = average_runs(scores)
(OUT / "rata_rata_seed.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
command = [sys.executable, "src/evaluate.py", "--gold", "data/annotations/final_v1/gold.csv",
           "--ids-split", "gold_test", "--out", str(OUT / "evaluasi")]
for spec in pred_args:
    command.extend(["--pred", spec])
subprocess.run(command, check=True)
print("Rata-rata dan simpangan baku tiga seed:")
print(json.dumps(summary, indent=2))
archive = shutil.make_archive("/content/hasil_indobert_lengkap", "zip", OUT)
files.download(archive)
print("Unduh hasil_indobert_lengkap.zip dan berikan berkas tersebut untuk memperbarui laporan.")
